# Build Repair Agent Evaluation

This notebook runs W&B Weave evaluations for the automatic build repair agent.

## Setup

In [ ]:
import os

import weave

os.environ["WEAVE_PARALLELISM"] = "8"

PROJECT_NAME = "bugbug-build-repair-eval"
_ = weave.init(PROJECT_NAME)

FIREFOX_REPO = os.environ["FIREFOX_GIT_REPO"]

## Load Dataset

In [ ]:
dataset = weave.ref("build_repair_one_commit_eval").get()
print(f"Dataset has {len(dataset.rows)} examples")

## Configure Model

In [ ]:
from functools import cached_property

from bugbug.tools.build_repair.agent import AgentResponse, BuildFailure, BuildRepairTool
from bugbug.tools.build_repair.worktree import WorktreeManager


class BuildRepairModel(weave.Model):
    firefox_repo: str

    @cached_property
    def tool(self) -> BuildRepairTool:
        return BuildRepairTool.create()

    @cached_property
    def worktree_mgr(self) -> WorktreeManager:
        return WorktreeManager(self.firefox_repo)

    @weave.op()
    async def invoke(
        self,
        bug_id: int,
        pre_fix_bug: dict,
        gh_failure_commits: list[str],
        failures: list[dict],
        **kwargs,
    ) -> dict:
        wt_name = f"bug-{bug_id}"
        worktree_path = self.worktree_mgr.create(gh_failure_commits[0], wt_name)
        try:
            failure = BuildFailure(
                bug_id=bug_id,
                bug_title=pre_fix_bug.get("title"),
                bug_comments=pre_fix_bug.get("comments"),
                git_commit=gh_failure_commits[0],
                failure_tasks=[
                    f
                    for f in failures
                    if "build" in f["task_name"] and "test" not in f["task_name"]
                ],
            )
            result: AgentResponse = await self.tool.run(
                failure, worktree_path=worktree_path
            )
            return result.model_dump()
        except Exception as e:
            return {
                "error": str(e),
                "diff": "",
                "summary": "",
                "analysis": "",
                "cost_usd": 0,
                "num_turns": 0,
                "local_build_passed": None,
                "try_build_passed": None,
                "lando_job_id": None,
                "treeherder_url": None,
            }
        finally:
            self.worktree_mgr.cleanup(wt_name)


model = BuildRepairModel(firefox_repo=FIREFOX_REPO)

## Run Evaluation

In [ ]:
from bugbug.tools.build_repair.scorer import (
    BasicMetricsScorer,
    BuildPassRateScorer,
    LLMFixMatchingScorer,
)

evaluation = weave.Evaluation(
    dataset=dataset,
    scorers=[BasicMetricsScorer(), BuildPassRateScorer(), LLMFixMatchingScorer()],
)

results = await evaluation.evaluate(model)

## Visualizations

In [ ]:
import matplotlib.pyplot as plt

basic = results.get("BasicMetricsScorer", {})
build = results.get("BuildPassRateScorer", {})

metrics = {
    "success_rate": basic.get("success_rate", 0),
    "diff_rate": basic.get("diff_rate", 0),
    "local_build_pass_rate": build.get("local_build_pass_rate", 0),
    "try_build_pass_rate": build.get("try_build_pass_rate", 0),
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(metrics.keys(), metrics.values())
ax.set_ylim(0, 1)
ax.set_ylabel("Rate")
ax.set_title("Build Repair Evaluation Results")
plt.tight_layout()
plt.show()

print(f"Total cost: ${basic.get('total_cost_usd', 0):.2f}")
print(f"Avg cost per example: ${basic.get('avg_cost_usd', 0):.2f}")

In [ ]:
print(f"Basic metrics: {basic}")
print(f"Build pass rates: {build}")

## 7. View in W&B

Visit [W&B Weave](https://wandb.ai) to see detailed traces, compare evaluations, and explore individual predictions.